## NOT RUN — future work; no numbers in the paper come from this script
Paper: "The Limits of Query Monitoring for Distributed Model Extraction".

**NOT RUN — future work; no numbers in the paper come from this script; it implements open tests listed in §11 'Limitations and open tests'.**
Specifically, it implements the §11 future-work item "distillation from a live teacher with more seeds, students and tasks"
(§11: the frontier rests on one 360M student, one benchmark, answers from datasets rather than a live teacher, and two seeds).
It also depends on outputs of Step 36, which was not run either. The predictions P37a–P37f below were written in this
notebook but never evaluated; they do not appear in App. P (prediction log). There is no reproduction table.

Step 37 —The capability side of the frontier with a live teacher, five seeds and the full GSM8K test
**Kaggle:** GPU (T4/P100) + Internet → **Run All** (≈6–7 h; resumable — results are checkpointed after every fine-tune, so a rerun skips finished runs). Attach Step 36's output (it contains `step36_full_prompts.pkl`) as an input dataset; the notebook finds it under `/kaggle/input`. `QUICK=True` first (≈30 min).

**Why.** The covert end of the frontier rested on one 360M student, 400 GSM8K problems (14 solved at baseline), two seeds and answers taken from datasets. Here a **live teacher** (Qwen2.5-1.5B-Instruct, greedy) answers every campaign prompt, the student (SmolLM2-360M, full fine-tuning, settings of Steps 19–32) is trained with **five seeds**, and every model is scored on the **full GSM8K test set (1,319 problems)**, with paired bootstrap intervals over problems and seeds for every frontier point. A second, smaller student (SmolLM2-135M) is added for the dataset and tilted campaigns.

Campaigns at 3,000 queries: `dataset_ds` (MetaMathQA GSM prompts with dataset answers, the earlier setting), `dataset_live` (same prompts, teacher answers), `padded_live` (half those prompts, half off-task WildChat prompts), `tilted_live` (half on-task WildChat prompts from the attacker pool, half off-task), `written_live` (half LLM rewrites in the users' voice that passed the attacker's discriminator, half off-task).

| ID | Prediction (written in this notebook only; never run, not evaluated, not in the paper's App. P) |
|---|---|
| **P37a** | With a live teacher the student is in the extraction regime: `dataset_live` gains ≥ 5 points |
| **P37b** | Padding costs ≥ 20% of the `dataset_live` gain, with the paired 95% interval of the difference excluding zero |
| **P37c** | The tilted campaign keeps ≤ 40% of the `dataset_live` gain |
| **P37d** | The written campaign keeps ≤ 50% of the `dataset_live` gain |
| **P37e** | Precision: seed standard deviation ≤ 1.5 points, and the paired interval of `dataset_live` minus `tilted_live` has half-width ≤ 2 points |
| **P37f** | Second student (SmolLM2-135M): if `dataset_live` gains ≥ 2 points, the tilted campaign keeps ≤ 50% of it |

In [ ]:
# ===== Cell 1: configuration and inputs =====
import math, json, time, os, gc, re, pickle, glob, sys, subprocess
import numpy as np
QUICK = False
SEED = 9091
OUTDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
import torch
DEV = torch.device("cuda"); assert torch.cuda.is_available(), "GPU required"
TEACHER, STUDENTS = "Qwen/Qwen2.5-1.5B-Instruct", {"smol360": "HuggingFaceTB/SmolLM2-360M", "smol135": "HuggingFaceTB/SmolLM2-135M"}
if QUICK: C = dict(BUD=120, SEEDS=[0, 1], N_EVAL=60, MAX_NEW=160, TEACH_NEW=160, TEACH_BS=16, BS=16, GEN_BS=32, MAX_LEN=384, SEEDS2=[0])
else:     C = dict(BUD=3000, SEEDS=[0, 1, 2, 3, 4], N_EVAL=1319, MAX_NEW=320, TEACH_NEW=384, TEACH_BS=48, BS=16, GEN_BS=64, MAX_LEN=512, SEEDS2=[0, 1, 2])
TAG = "quick" if QUICK else "full"
def find(name):
    for p in [f"{OUTDIR}/{name}"] + glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        if os.path.exists(p): return p
    return None
fp = find(f"step36_{TAG}_prompts.pkl") or find("step36_full_prompts.pkl")
assert fp, "Attach Step 36's output (step36_*_prompts.pkl) as an input dataset."
with open(fp, "rb") as fh: P = pickle.load(fh)
GSM_ROWS, ON, OFF, WRITTEN = P["gsm_rows"], P["on_task"], P["off_task"], P["written"]
print(f"inputs: MetaMathQA GSM rows {len(GSM_ROWS)}, on-task pool prompts {len(ON)}, off-task {len(OFF)}, written (passing) {len(WRITTEN)}")
RES_PATH = f"{OUTDIR}/step37_{TAG}_results.json"; RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
CORR_PATH = f"{OUTDIR}/step37_{TAG}_correct.npz"; CORR = dict(np.load(CORR_PATH)) if os.path.exists(CORR_PATH) else {}

In [ ]:
# ===== Cell 2: campaign prompt sets (fixed across seeds) =====
rng = np.random.default_rng(SEED); B = C["BUD"]; h = B // 2
mm_ix = rng.choice(len(GSM_ROWS), B, replace=False); off_ix = rng.choice(len(OFF), B, replace=False)
on_ix = rng.choice(len(ON), h, replace=len(ON) < h); wr_ix = rng.choice(len(WRITTEN), h, replace=len(WRITTEN) < h)
PROMPTS = {
    "dataset": [GSM_ROWS[i][0] for i in mm_ix],
    "padded": [GSM_ROWS[i][0] for i in mm_ix[:h]] + [OFF[i] for i in off_ix[: B - h]],
    "tilted": [ON[i] for i in on_ix] + [OFF[i] for i in off_ix[: B - h]],
    "written": [WRITTEN[i][0] for i in wr_ix] + [OFF[i] for i in off_ix[: B - h]],
}
DS_ANS = [GSM_ROWS[i][1] for i in mm_ix]
UNIQUE = {"tilted_on": int(len(set(on_ix))), "written_on": int(len(set(wr_ix)))}
print(f"unique on-task prompts: tilted {UNIQUE['tilted_on']}/{h}, written {UNIQUE['written_on']}/{h}")

In [ ]:
# ===== Cell 3: the live teacher answers every prompt (cached) =====
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
TP = f"{OUTDIR}/step37_{TAG}_teacher.pkl"
SYS = "You are a helpful assistant. When a question has a numerical answer, reason step by step and end with 'The answer is: <number>'."
if os.path.exists(TP):
    with open(TP, "rb") as fh: TEACH = pickle.load(fh)
else:
    t0 = time.time(); ttok = AutoTokenizer.from_pretrained(TEACHER); ttok.padding_side = "left"
    tm = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=torch.float16).to(DEV).eval()
    allp = sorted(set(p for v in PROMPTS.values() for p in v)); TEACH = {}
    order = np.argsort([len(p) for p in allp])                                             # length-sorted batches
    for s0 in range(0, len(allp), C["TEACH_BS"]):
        batch = [allp[i] for i in order[s0:s0 + C["TEACH_BS"]]]
        msgs = [ttok.apply_chat_template([{"role": "system", "content": SYS}, {"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in batch]
        ids = ttok(msgs, return_tensors="pt", padding=True, truncation=True, max_length=768).to(DEV)
        with torch.no_grad(): out = tm.generate(**ids, max_new_tokens=C["TEACH_NEW"], do_sample=False, pad_token_id=ttok.eos_token_id)
        for p, o in zip(batch, out): TEACH[p] = ttok.decode(o[ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()[:2000]
        if (s0 // C["TEACH_BS"]) % 20 == 0: print(f"  [teacher] {len(TEACH)}/{len(allp)} ({time.time()-t0:.0f}s)")
    del tm; gc.collect(); torch.cuda.empty_cache()
    with open(TP, "wb") as fh: pickle.dump(TEACH, fh)
PAIRS = {"dataset_ds": list(zip(PROMPTS["dataset"], DS_ANS))}
for k in ["dataset", "padded", "tilted", "written"]: PAIRS[k + "_live"] = [(p, TEACH[p]) for p in PROMPTS[k]]
print({k: len(v) for k, v in PAIRS.items()})

In [ ]:
# ===== Cell 4: student training and GSM8K evaluation (settings of Steps 19-32) =====
from datasets import load_dataset
fmtq = lambda q: f"Question: {q.strip()}\nAnswer:"
def train_full(student, pairs, seed, tok, lr=5e-5):
    torch.manual_seed(seed); model = AutoModelForCausalLM.from_pretrained(student, torch_dtype=torch.float32).to(DEV)
    model.gradient_checkpointing_enable(); model.config.use_cache = False
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0); steps = max(1, math.ceil(len(pairs) / C["BS"]))
    sch = get_linear_schedule_with_warmup(opt, max(1, steps // 30), steps); scaler = torch.cuda.amp.GradScaler()
    tok.padding_side = "right"; model.train(); order = np.random.default_rng(seed).permutation(len(pairs))
    MICRO, ACC = 4, max(1, C["BS"] // 4); starts = list(range(0, len(pairs), MICRO))
    for i, s0 in enumerate(starts):
        batch = [pairs[k] for k in order[s0:s0 + MICRO]]
        enc = tok([fmtq(q) + " " + a.strip() + tok.eos_token for q, a in batch], return_tensors="pt", padding=True, truncation=True, max_length=C["MAX_LEN"]).to(DEV)
        labels = enc["input_ids"].clone(); labels[enc["attention_mask"] == 0] = -100
        for k, (q, _) in enumerate(batch): labels[k, : len(tok(fmtq(q))["input_ids"])] = -100
        with torch.autocast("cuda", dtype=torch.float16): loss = model(**enc, labels=labels).loss / ACC
        scaler.scale(loss).backward(); del enc, labels, loss
        if (i + 1) % ACC == 0 or i == len(starts) - 1:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()
    model = model.half(); model.config.use_cache = True; return model
@torch.no_grad()
def generate(model, tok, questions, max_new):
    model.eval(); tok.padding_side = "left"; outs = []
    for i in range(0, len(questions), C["GEN_BS"]):
        enc = tok([fmtq(q) for q in questions[i:i + C["GEN_BS"]]], return_tensors="pt", padding=True).to(DEV)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).split("\nQuestion:")[0] for o in out]
    return outs
def last_number(t):
    m = re.search(r"answer is:?\s*\$?(-?[\d,]*\.?\d+)", t); s = m.group(1) if m else (re.findall(r"-?[\d,]*\.?\d+", t) or [None])[-1]
    try: return float(s.replace(",", "")) if s else None
    except ValueError: return None
gsm = load_dataset("openai/gsm8k", "main", split="test").select(range(C["N_EVAL"])); GQ = list(gsm["question"]); GOLD = [float(a.split("####")[-1].strip().replace(",", "")) for a in gsm["answer"]]
def gsm_correct(model, tok): return np.array([p is not None and abs(p - g) < 1e-4 for p, g in zip([last_number(t) for t in generate(model, tok, GQ, C["MAX_NEW"])], GOLD)], np.int8)
def save():
    json.dump(RES, open(RES_PATH, "w")); np.savez(CORR_PATH, **CORR)

In [ ]:
# ===== Cell 5: runs (checkpointed) =====
t0 = time.time()
plan = [("smol360", k, sd) for k in ["dataset_live", "tilted_live", "padded_live", "written_live", "dataset_ds"] for sd in C["SEEDS"]]
plan += [("smol135", k, sd) for k in ["dataset_live", "tilted_live"] for sd in C["SEEDS2"]]
for stu in ["smol360", "smol135"]:
    key = f"{stu}|base"
    if key not in CORR:
        tok = AutoTokenizer.from_pretrained(STUDENTS[stu]); tok.pad_token = tok.pad_token or tok.eos_token
        bm = AutoModelForCausalLM.from_pretrained(STUDENTS[stu], dtype=torch.float16).to(DEV); CORR[key] = gsm_correct(bm, tok); del bm; torch.cuda.empty_cache(); save()
    print(f"[base] {stu}: {CORR[key].mean():.4f}")
for stu, camp, sd in plan:
    key = f"{stu}|{camp}|{sd}"
    if key in CORR: continue
    tok = AutoTokenizer.from_pretrained(STUDENTS[stu]); tok.pad_token = tok.pad_token or tok.eos_token
    model = train_full(STUDENTS[stu], PAIRS[camp], sd, tok); CORR[key] = gsm_correct(model, tok); del model; gc.collect(); torch.cuda.empty_cache()
    RES[key] = float(CORR[key].mean()); save()
    print(f"[run] {key}: {RES[key]:.4f} (gain {100 * (RES[key] - CORR[f'{stu}|base'].mean()):+.1f})  ({time.time()-t0:.0f}s)")

In [ ]:
# ===== Cell 6: frontier points with paired bootstrap intervals; predictions and verdict =====
def runs(stu, camp, seeds): return np.array([CORR[f"{stu}|{camp}|{s}"] for s in seeds if f"{stu}|{camp}|{s}" in CORR], float)
def gain(stu, camp, seeds, B_=2000, rng=np.random.default_rng(3)):
    X = runs(stu, camp, seeds); base = CORR[f"{stu}|base"].astype(float)
    if len(X) == 0: return None
    d = X.mean(0) - base; bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(B_)]
    return dict(gain=100 * float(d.mean()), ci=[100 * float(np.quantile(bs, .025)), 100 * float(np.quantile(bs, .975))], seed_sd=100 * float(X.mean(1).std(ddof=1)) if len(X) > 1 else float("nan"), n_seeds=len(X))
def paired(stu, a, b, seeds, B_=2000, rng=np.random.default_rng(4)):
    Xa, Xb = runs(stu, a, seeds), runs(stu, b, seeds)
    if len(Xa) == 0 or len(Xb) == 0: return None
    d = Xa.mean(0) - Xb.mean(0); bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(B_)]
    return dict(diff=100 * float(d.mean()), ci=[100 * float(np.quantile(bs, .025)), 100 * float(np.quantile(bs, .975))])
G = {(stu, c): gain(stu, c, C["SEEDS"] if stu == "smol360" else C["SEEDS2"]) for stu in STUDENTS for c in ["dataset_ds", "dataset_live", "padded_live", "tilted_live", "written_live"]}
for k, v in G.items():
    if v: print(f"[gain] {k[0]} {k[1]:>13}: {v['gain']:+.1f} [{v['ci'][0]:+.1f}, {v['ci'][1]:+.1f}] seed sd {v['seed_sd']:.2f} ({v['n_seeds']} seeds)")
dl = G[("smol360", "dataset_live")]["gain"]
keep = lambda c: G[("smol360", c)]["gain"] / dl if G[("smol360", c)] and dl > 0 else float("nan")
pp = paired("smol360", "dataset_live", "padded_live", C["SEEDS"]); pt = paired("smol360", "dataset_live", "tilted_live", C["SEEDS"]); pw = paired("smol360", "dataset_live", "written_live", C["SEEDS"])
print(f"P37a dataset_live gain {dl:+.1f} (>= 5)")
print(f"P37b padded keeps {keep('padded_live'):.2f} of the gain (cost >= 0.20); paired dataset-padded {pp['diff']:+.1f} {[round(x, 1) for x in pp['ci']]} (excludes 0)")
print(f"P37c tilted keeps {keep('tilted_live'):.2f} (<= 0.40); P37d written keeps {keep('written_live'):.2f} (<= 0.50); paired dataset-written {pw['diff']:+.1f} {[round(x, 1) for x in pw['ci']]}")
print(f"P37e seed sd (max over campaigns) {max(v['seed_sd'] for k, v in G.items() if v and k[0] == 'smol360'):.2f} (<= 1.5); paired dataset-tilted {pt['diff']:+.1f} {[round(x, 1) for x in pt['ci']]} half-width {(pt['ci'][1] - pt['ci'][0]) / 2:.2f} (<= 2)")
g2d, g2t = G[("smol135", "dataset_live")], G[("smol135", "tilted_live")]
if g2d and g2t: print(f"P37f SmolLM2-135M: dataset_live {g2d['gain']:+.1f}, tilted keeps {g2t['gain'] / g2d['gain'] if g2d['gain'] > 0 else float('nan'):.2f} (<= 0.5 if dataset >= 2)")
print("=" * 90); print("STEP 37 VERDICT REPORT"); print("=" * 90)
print("[BASE]", {s: float(CORR[f'{s}|base'].mean()) for s in STUDENTS if f'{s}|base' in CORR})
print("[GAIN]", json.dumps({f"{k[0]}|{k[1]}": v for k, v in G.items()}))
print("[PAIRED]", json.dumps(dict(padded=pp, tilted=pt, written=pw))); print("[UNIQUE]", UNIQUE); print("=" * 90)